# Visualize a Mickelin GNS run

Reads one resampled run from the Zarr dataset produced by
`datagen/mickelin/scripts/postprocess.py` and renders snapshots,
Hovmöller diagrams, and animations of the vorticity field on the sphere.

Dataset layout: `(time, field, lat, lon)` with `field = ['vorticity']`.
Time is in τ (the Mickelin script uses τ-units for both `snapshot_dt` and
`stop_sim_time`, and the Zarr metadata records that as seconds — but the
configs set `tau = 1` so the numbers are effectively τ).

## Environment

Needs `xarray`, `zarr`, `numpy`, `matplotlib`, `cartopy`, `imageio[ffmpeg]`.
All ship with the `datagen` project env. Add `ipykernel` / `jupyterlab`
if not yet installed:

```bash
uv add --project datagen ipykernel jupyterlab
uv run --project datagen jupyter lab
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# Point this at any run_XXXX.zarr produced by the Mickelin sweep.
DATA_PATH = Path("/scicore/home/dokman0000/GROUP/PDEDatasets/SphericalPDEs/mickelin-gns/processed/run_0000.zarr")
DATA_PATH = Path("/scicore/home/dokman0000/GROUP/PDEDatasets/SphericalPDEs/mickelin-gns/processed/run_0080.zarr")
#DATA_PATH = Path("/scicore/home/dokman0000/GROUP/PDEDatasets/SphericalPDEs/mickelin-gns/processed/run_0360.zarr")
#DATA_PATH = Path("/scicore/home/dokman0000/GROUP/PDEDatasets/SphericalPDEs/mickelin-gns/processed/run_0440.zarr")

ds = xr.open_zarr(DATA_PATH)
ds

In [ ]:
# Quick parameter + shape summary.
print("Dims:", dict(ds.sizes))
print("Time range:", float(ds.time.min()), "..", float(ds.time.max()), "τ")
print("Lat range: ", float(ds.lat.min()), "..", float(ds.lat.max()), "deg")
print("Lon range: ", float(ds.lon.min()), "..", float(ds.lon.max()), "deg")
print("Parameters:")
for k, v in ds.attrs.items():
    if k.startswith("param_"):
        print(f"  {k[6:]:14s} = {v}")

## Initial state (post-transient)

Postprocess discards the first 30 τ of spin-up and rebases time to 0,
so the first kept snapshot already shows fully developed turbulence —
expect a sea of vortices at the characteristic scale set by `Λ`.

In [ ]:
lat = ds.lat.values
lon = ds.lon.values
vort_all = ds.fields.sel(field="vorticity").values  # (time, lat, lon)
vmax = float(np.nanmax(np.abs(vort_all))) * 0.7

fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
im = ax.pcolormesh(lon, lat, vort_all[0], cmap="RdBu_r",
                   vmin=-vmax, vmax=vmax, shading="auto")
ax.set_title(f"vorticity at t = {float(ds.time.isel(time=0)):.2f} τ")
ax.set_xlabel("lon [deg]")
ax.set_ylabel("lat [deg]")
plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02, label="vorticity")
plt.show()

## Time evolution of vorticity

Six panels evenly spaced through the kept window. Vortices should
drift, merge, and reorganize while the spectrum stays peaked around
the GNS preferred scale.

In [ ]:
n_times = ds.sizes["time"]
panels = np.linspace(0, n_times - 1, 6, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(18, 8), constrained_layout=True, sharex=True, sharey=True)
for ax, ti in zip(axes.flat, panels):
    im = ax.pcolormesh(lon, lat, vort_all[ti], cmap="RdBu_r",
                       vmin=-vmax, vmax=vmax, shading="auto")
    ax.set_title(f"t = {float(ds.time.isel(time=ti)):.2f} τ")
    ax.set_xlabel("lon [deg]")
    ax.set_ylabel("lat [deg]")
fig.colorbar(im, ax=axes, fraction=0.015, pad=0.02, label="vorticity")
fig.suptitle("Mickelin GNS vorticity evolution")
plt.show()

## Zonal-mean Hovmöller diagram

Zonal mean of the vorticity over longitude as a function of `(time, lat)`.
GNS turbulence on the sphere is statistically isotropic, so this should
look like noise around zero with no persistent banded structure.

In [ ]:
vort_zm = ds.fields.sel(field="vorticity").mean("lon")  # (time, lat)
t_tau = ds.time.values
vmax_zm = float(np.abs(vort_zm).max())

fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
im = ax.pcolormesh(t_tau, lat, vort_zm.T, cmap="RdBu_r",
                   vmin=-vmax_zm, vmax=vmax_zm, shading="auto")
ax.set_title("Zonal-mean vorticity")
ax.set_xlabel("time [τ]")
ax.set_ylabel("lat [deg]")
fig.colorbar(im, ax=ax, label="<vorticity>_lon")
plt.show()

## Globally-integrated diagnostics

Area-weighted RMS and global max of `|vorticity|`. Statistically
stationary turbulence shows fluctuations around a roughly constant
level. A drift would suggest the post-transient window is still in
spin-up; a blow-up signals a solver problem.

In [ ]:
lat_rad = np.deg2rad(ds.lat.values)
w = np.cos(lat_rad)
w = w / w.sum()

rms_vort = np.sqrt((vort_all ** 2).mean(axis=-1) @ w)  # (time,)
max_vort = np.abs(vort_all).max(axis=(-1, -2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(t_tau, rms_vort)
axes[0].set_xlabel("time [τ]")
axes[0].set_ylabel("RMS vorticity")
axes[0].set_title("Area-weighted RMS vorticity")
axes[1].plot(t_tau, max_vort)
axes[1].set_xlabel("time [τ]")
axes[1].set_ylabel("max |vorticity|")
axes[1].set_title("Global max |vorticity|")
plt.show()

## Vorticity animation (flat MP4)

Same canvas-grab pattern as the Galewsky notebook. `imageio[ffmpeg]`
bundles its own ffmpeg, no system install needed.

In [ ]:
import imageio.v3 as iio
from tqdm import trange

fps = 20
dpi = 100
output_path = Path("mickelin_vorticity.mp4")

frames = []
for i in trange(vort_all.shape[0]):
    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True, dpi=dpi)
    mesh = ax.pcolormesh(lon, lat, vort_all[i], cmap="RdBu_r",
                          vmin=-vmax, vmax=vmax, shading="auto")
    ax.set_title(f"t = {float(ds.time.isel(time=i)):.2f} τ")
    ax.set_xlabel("lon [deg]")
    ax.set_ylabel("lat [deg]")
    fig.colorbar(mesh, ax=ax, label="vorticity")

    fig.canvas.draw()
    w_px, h_px = fig.canvas.get_width_height()
    frame = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h_px, w_px, 4)
    frame = frame[: h_px - (h_px % 2), : w_px - (w_px % 2), :3]
    frames.append(frame)
    plt.close(fig)

iio.imwrite(
    str(output_path),
    np.stack(frames, axis=0),
    fps=fps,
    codec="libx264",
    pixelformat="yuv420p",
)
print(f"Wrote {output_path}  ({len(frames)} frames @ {fps} fps)")

## Vorticity animation (spherical, rotating globe)

Cartopy orthographic projection with a slow rotation around the
central longitude over the course of the run. This is usually the
most informative view for GNS turbulence on the sphere — the flat
projection distorts the polar vortices.

In [ ]:
import cartopy.crs as ccrs
from tqdm import trange

fps = 20
dpi = 100
output_path = Path("mickelin_vorticity_spherical.mp4")

num_frames = vort_all.shape[0]
frames = []

for i in trange(num_frames):
    fig = plt.figure(figsize=(8, 8), constrained_layout=True, dpi=dpi)

    cent_lon = (i / num_frames) * 360.0 - 180.0
    ax = fig.add_subplot(
        1, 1, 1,
        projection=ccrs.Orthographic(central_longitude=cent_lon, central_latitude=20.0),
    )
    ax.set_global()
    ax.gridlines(alpha=0.3, linestyle="--")

    mesh = ax.pcolormesh(
        lon, lat, vort_all[i],
        cmap="RdBu_r",
        vmin=-vmax, vmax=vmax,
        shading="auto",
        transform=ccrs.PlateCarree(),
    )
    ax.set_title(f"Mickelin GNS vorticity | t = {float(ds.time.isel(time=i)):.2f} τ")
    fig.colorbar(mesh, ax=ax, shrink=0.6, label="vorticity", pad=0.05)

    fig.canvas.draw()
    w_px, h_px = fig.canvas.get_width_height()
    frame = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h_px, w_px, 4)
    frame = frame[: h_px - (h_px % 2), : w_px - (w_px % 2), :3]
    frames.append(frame)
    plt.close(fig)

iio.imwrite(
    str(output_path),
    np.stack(frames, axis=0),
    fps=fps,
    codec="libx264",
    pixelformat="yuv420p",
)
print(f"Wrote {output_path}  ({len(frames)} frames @ {fps} fps)")